In [1]:
import findspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, isnan, when, count, round as spark_round
from pyspark.sql.types import DoubleType, FloatType

In [2]:
findspark.init()

In [3]:
spark = SparkSession.builder \
    .appName("US Accidents Analysis - Local Laptop") \
    .master("local[*]") \
    .config("spark.driver.memory", "6g") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.default.parallelism", "16") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryoserializer.buffer.max", "512m") \
    .getOrCreate()

spark

In [4]:
Data_Path = "Data/Full_Data"

In [5]:
df = spark.read \
    .option("mergeSchema", "false") \
    .parquet(Data_Path)

In [7]:
df = df.repartition(16)

In [8]:
df.printSchema()

root
 |-- ID: string (nullable = true)
 |-- Severity: integer (nullable = true)
 |-- Start_Time: timestamp (nullable = true)
 |-- End_Time: timestamp (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)
 |-- End_Lat: double (nullable = true)
 |-- End_Lng: double (nullable = true)
 |-- Distance(mi): double (nullable = true)
 |-- Description: string (nullable = true)
 |-- Street: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Timezone: string (nullable = true)
 |-- Airport_Code: string (nullable = true)
 |-- Weather_Timestamp: timestamp (nullable = true)
 |-- Temperature(F): double (nullable = true)
 |-- Wind_Chill(F): double (nullable = true)
 |-- Humidity(%): double (nullable = true)
 |-- Pressure(in): double (nullable = true)
 |-- Visibility(mi): double (nullable = true

In [9]:
print("This is Snap of the full dataset:")
df.show(5)

This is Snap of the full dataset:
+---------+--------+-------------------+-------------------+---------+-----------+-----------------+----------+------------+--------------------+-----------------+-------------+------------+-----+----------+-------+-----------+------------+-------------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+-----------------+-------+-----+--------+--------+--------+-------+-------+----------+-------+-----+---------------+--------------+------------+--------------+--------------+-----------------+---------------------+
|       ID|Severity|         Start_Time|           End_Time|Start_Lat|  Start_Lng|          End_Lat|   End_Lng|Distance(mi)|         Description|           Street|         City|      County|State|   Zipcode|Country|   Timezone|Airport_Code|  Weather_Timestamp|Temperature(F)|Wind_Chill(F)|Humidity(%)|Pressure(in)|Visibility(mi)|Wind_Direction|Wind_Speed(mph)|Precipitation

In [10]:
data_records = df.count()
data_columns = len(df.columns)
print(f"Total number of records in the dataset: {data_records}")
print(f"Total number of columns in the dataset: {data_columns}")

Total number of records in the dataset: 10573736
Total number of columns in the dataset: 45


In [13]:
print("This is the summary statistics of the dataset:")
df.describe().show()

This is the summary statistics of the dataset:
+-------+--------+------------------+-----------------+------------------+-----------------+------------------+------------------+--------------------+--------------------+----------+---------+--------+------------------+--------+----------+------------+------------------+------------------+-----------------+------------------+------------------+--------------+-----------------+--------------------+------------------+--------------+--------------+-----------------+---------------------+
|summary|      ID|          Severity|        Start_Lat|         Start_Lng|          End_Lat|           End_Lng|      Distance(mi)|         Description|              Street|      City|   County|   State|           Zipcode| Country|  Timezone|Airport_Code|    Temperature(F)|     Wind_Chill(F)|      Humidity(%)|      Pressure(in)|    Visibility(mi)|Wind_Direction|  Wind_Speed(mph)|   Precipitation(in)| Weather_Condition|Sunrise_Sunset|Civil_Twilight|Nautical_T

In [14]:
def null_analysis(df, name):
    print(f"\n=== Null Analysis: {name} ===")
    total = df.count()
    
    null_exprs = []
    for field in df.schema.fields:
        c = field.name
        # isnan() is only valid for numeric (Double/Float) columns
        # For all other types, just check isNull()
        if isinstance(field.dataType, (DoubleType, FloatType)):
            condition = col(c).isNull() | isnan(c)
        else:
            condition = col(c).isNull()
        
        null_exprs.append(count(when(condition, c)).alias(c))
    
    null_counts = df.select(null_exprs)
    
    null_df = null_counts.toPandas().T.reset_index()
    null_df.columns = ["column", "null_count"]
    null_df["null_pct"] = (null_df["null_count"] / total * 100).round(2)
    null_df = null_df[null_df["null_count"] > 0].sort_values("null_pct", ascending=False)
    
    if null_df.empty:
        print("No nulls found.")
    else:
        print(null_df.to_string(index=False))

In [15]:
null_analysis(df, "Full Dataset")


=== Null Analysis: Full Dataset ===
               column  null_count  null_pct
              End_Lat     3402762     32.18
              End_Lng     3402762     32.18
    Precipitation(in)     2753044     26.04
        Wind_Chill(F)     2468662     23.35
      Wind_Speed(mph)      729177      6.90
       Wind_Direction      248981      2.35
       Visibility(mi)      247644      2.34
          Humidity(%)      247236      2.34
    Weather_Condition      244095      2.31
       Temperature(F)      233127      2.20
         Pressure(in)      199879      1.89
    Weather_Timestamp      170964      1.62
         Airport_Code       32184      0.30
       Sunrise_Sunset       26113      0.25
    Nautical_Twilight       26113      0.25
       Civil_Twilight       26113      0.25
Astronomical_Twilight       26113      0.25
             Timezone       11467      0.11
               Street       10871      0.10
              Zipcode        3234      0.03
                 City         390      

In [16]:
total_dec = df.count()
df.groupBy("Severity") \
    .agg(count("*").alias("count")) \
    .withColumn("percentage", spark_round(col("count") / total_dec * 100, 2)) \
    .orderBy("Severity") \
    .show()

+--------+-------+----------+
|Severity|  count|percentage|
+--------+-------+----------+
|       1|  93419|      0.88|
|       2|8689972|     82.18|
|       3|1454442|     13.76|
|       4| 335903|      3.18|
+--------+-------+----------+



In [18]:
duplicates = df.groupBy(df.columns) \
    .count() \
    .filter("count > 1")
num_duplicates = duplicates.count()
print(f"Number of duplicate records: {num_duplicates}")

Number of duplicate records: 0
